# Extract training samples
* Download low-tide cloud free satellite iamges closest to the UAV image collection
* Sample the satellite image bands where appromximately a single UAV class

In [13]:
import pathlib
import numpy
import dask.distributed

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:59754/status,
Dashboard: http://127.0.0.1:59754/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:59756,Workers: 0
Dashboard: http://127.0.0.1:59754/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:59780,Total threads: 2
Dashboard: http://127.0.0.1:59783/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:59759,


# Values to edit

In [15]:
max_cloud_covers = [1, 5, 0, 10] # percentage
tide_deltas = {1: {"hrs": 0, "mins": 30}, 
               2: {"hrs": 0, "mins": 45}, 
               3: {"hrs": 1, "mins": 0}}
low_tide_delta_hrs = 0
low_tide_delta_mins = 45
lowtide_search_range = 60 # days - +/- this devided by 2

In [16]:
site_names = ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

# Cells to run

In [17]:
data_path = utils.get_data_path()
utils.create_data_folders()

training_labels_file = data_path / "ELF24505_ClassificationClasses.txt"
servey_dates_file = data_path / "ELF24505_SurveyDates.csv" 
uav_folder = data_path / "classified_uav"

In [ ]:
for tide_delta in tide_deltas.values():
    for max_cloud_cover in max_cloud_covers:
        print(f"\n\nMax cloud percentage {max_cloud_cover}, Low tide delta {tide_delta['hrs']}hrs and {tide_delta['mins']}mins")
        for site_name in site_names[4:9]:
            sampling.get_site_satellite(
                site_name=site_name,
                survey_dates_file=servey_dates_file,
                lowtide_search_range=lowtide_search_range,
                low_tide_delta_hrs=tide_delta["hrs"],
                low_tide_delta_mins=tide_delta["mins"],
                uav_folder=uav_folder,
                max_cloud_cover=max_cloud_cover,
                apply_anomaly_filter=True,
                n_mad=3,
            )
            print(f"Save RGB for {site_name}")
            satellite_images_path = utils.get_training_satellite_path(site_name=site_name,  low_tide_delta_hrs=low_tide_delta_hrs,
                                                             low_tide_delta_mins=low_tide_delta_mins, max_cloud_cover=max_cloud_cover,
                                                             filtered=True)
            utils.rgb_from_satellite(satellite_data_path=satellite_images_path)



Max cloud percentage 1, Low tide delta 0hrs and 30mins
Site Duvauchelle
	Save satellite image of the site around lowtide without cloud
	Satellite date range 2025-01-30/2025-03-31
	Time from low tide: 0:25:28.975000 for S2C_MSIL2A_20250228T221631_R129_T59GPM_20250301T003411
	Time from low tide: 0:29:11.025000 for S2C_MSIL2A_20250313T222611_R029_T59GPM_20250314T010517
	Low tide tiles: 2 from a total lowish cloud cover tiles of 22
	No cloud tiles: 1 from the low tide tiles of 2. Cloud percentages [  0.1 100. ] Kept [ True False]
		2025-02-28: B04=0.59, B03=1.20, B02=2.70 -> kept
	Anomaly filter: 1 of 1 dates kept for site Duvauchelle (n_mad=3)
Save RGB for Duvauchelle
Satellite data path C:\Local\repos\seagrass-detection\data\training\tide_45mins_cloud_1\satellite_images_filtered\Duvauchelle_sentinel-2.nc does not exist.
Site Robinsons
	Save satellite image of the site around lowtide without cloud
	Satellite date range 2025-01-31/2025-04-01
	Time from low tide: 0:25:28.975000 for S2C_MS